# Phase 1 Final Feature Matrix Materialization

Notebook fix marker: `phase1_final_feature_matrix_v3_source_signal_diagnostics_20260421`

Purpose: materialize final scalp EEG feature values from the reviewed final feature manifest and final LOSO split so final comparator runners have a concrete input matrix.

Scientific integrity rule: this notebook does **not** train final comparators, compute comparator metrics, create logits, run controls/calibration/influence, audit runtime comparator logs, or open claims. If extraction prerequisites fail, the CLI writes a blocked artifact instead of a fake matrix, and Cell 7 prints a diagnostic payload before stopping. Non-finite signal samples or missing source channels are blockers, not values to impute silently.


## Technical Basis And Boundary

This notebook follows repository contracts:

- `configs/phase1/final_feature_matrix.json`: feature matrix materialization rules and non-claim boundary.
- `final_split_manifest.json`: reviewed LOSO subject split; materialization must not change folds.
- `final_feature_manifest.json`: reviewed feature schema/provenance; feature names and planned row count are source of truth.
- `final_leakage_audit.json`: manifest-level fit-scope audit; runtime comparator logs remain missing until final runners execute.
- `phase1_final_comparator_runner_readiness`: confirms final comparator outputs are still missing and smoke artifacts are not promoted.
- `src/phase1/final_feature_matrix.py`: durable CLI logic for extraction, validation, provenance and claim-state blocking.

Allowed interpretation: a final scalp feature matrix exists for downstream final comparator implementation if validation passes. Not allowed: using this matrix alone as decoder evidence or privileged-transfer evidence.


In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, signal extras preflight, and unit tests.

from pathlib import Path
import base64
import getpass
import importlib.util
import json
import os
import subprocess

NOTEBOOK_FIX_MARKER = 'phase1_final_feature_matrix_v3_source_signal_diagnostics_20260421'
print('Notebook fix marker:', NOTEBOOK_FIX_MARKER)

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()
INSTALL_SIGNAL_EXTRAS = True

def run(cmd, cwd=None, check=True, env=None):
    display = []
    for item in map(str, cmd):
        display.append('<redacted>' if 'Authorization: Basic' in item else item)
    print('$ ' + ' '.join(display))
    result = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

if IN_COLAB and not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, REPO_DIR])
elif REPO_DIR.exists():
    print(REPO_DIR)

run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

if importlib.util.find_spec('mne') is None:
    if not INSTALL_SIGNAL_EXTRAS:
        raise RuntimeError('Final feature matrix materialization requires mne/signal extras. Set INSTALL_SIGNAL_EXTRAS=True and rerun Cell 1.')
    env = os.environ.copy()
    env['INSTALL_SIGNAL_EXTRAS'] = '1'
    run(['bash', 'bootstrap/install_runtime.sh'], cwd=REPO_DIR, env=env)
else:
    print('mne is already available for EDF feature extraction.')

unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Do not materialize or review final feature matrix artifacts until tests pass.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Fixed paths and expected locked prereg identity.
# Update reviewed run paths only when newer upstream runs have been intentionally reviewed.

EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
FINAL_SPLIT_RUN = DRIVE_ROOT / 'artifacts/phase1_final_split_manifest/20260421T082928310517Z'
FINAL_FEATURE_RUN = DRIVE_ROOT / 'artifacts/phase1_final_feature_manifest/20260421T084604606845Z'
FINAL_LEAKAGE_RUN = DRIVE_ROOT / 'artifacts/phase1_final_leakage_audit/20260421T085953617562Z'
RUNNER_READINESS_RUN = DRIVE_ROOT / 'artifacts/phase1_final_comparator_runner_readiness/20260421T121130517494Z'
DATASET_ROOT = DRIVE_ROOT / 'data/ds004752'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_feature_matrix'

print('Prereg bundle:', PREREG_BUNDLE)
print('Final split run:', FINAL_SPLIT_RUN)
print('Final feature run:', FINAL_FEATURE_RUN)
print('Final leakage run:', FINAL_LEAKAGE_RUN)
print('Runner readiness run:', RUNNER_READINESS_RUN)
print('Dataset root:', DATASET_ROOT)
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Cell 3 - Validate prereg and upstream non-claim sources.

import hashlib


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

assert PREREG_BUNDLE.exists(), f'Missing prereg bundle: {PREREG_BUNDLE}'
bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_HASH, 'Prereg identity hash mismatch.'

split_summary = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_summary.json')
split_validation = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_validation.json')
split_manifest = read_json(FINAL_SPLIT_RUN / 'final_split_manifest.json')
feature_summary = read_json(FINAL_FEATURE_RUN / 'phase1_final_feature_manifest_summary.json')
feature_validation = read_json(FINAL_FEATURE_RUN / 'phase1_final_feature_manifest_validation.json')
feature_manifest = read_json(FINAL_FEATURE_RUN / 'final_feature_manifest.json')
leakage_summary = read_json(FINAL_LEAKAGE_RUN / 'phase1_final_leakage_audit_summary.json')
leakage_validation = read_json(FINAL_LEAKAGE_RUN / 'phase1_final_leakage_audit_validation.json')
leakage_audit = read_json(FINAL_LEAKAGE_RUN / 'final_leakage_audit.json')
readiness_summary = read_json(RUNNER_READINESS_RUN / 'phase1_final_comparator_runner_readiness_summary.json')
readiness_claim = read_json(RUNNER_READINESS_RUN / 'phase1_final_comparator_runner_claim_state.json')

print('Split:', split_summary.get('status'), split_summary.get('split_manifest_ready'))
print('Feature:', feature_summary.get('status'), feature_summary.get('feature_manifest_ready'), 'planned rows:', feature_manifest.get('n_event_rows_planned'))
print('Leakage:', leakage_summary.get('status'), leakage_summary.get('leakage_audit_ready'))
print('Runner readiness:', readiness_summary.get('status'), readiness_summary.get('upstream_manifests_ready'))

assert split_summary.get('status') == 'phase1_final_split_manifest_recorded'
assert split_summary.get('split_manifest_ready') is True
assert split_validation.get('status') == 'phase1_final_split_manifest_validation_passed'
assert split_validation.get('no_subject_overlap_between_train_and_test') is True
assert split_manifest.get('claim_ready') is False
assert split_manifest.get('smoke_artifacts_promoted') is False
assert feature_summary.get('status') == 'phase1_final_feature_manifest_recorded'
assert feature_summary.get('feature_manifest_ready') is True
assert feature_validation.get('status') == 'phase1_final_feature_manifest_validation_passed'
assert feature_manifest.get('contains_feature_matrix') is False
assert feature_manifest.get('contains_model_outputs') is False
assert feature_manifest.get('contains_metrics') is False
assert feature_manifest.get('claim_ready') is False
assert leakage_summary.get('status') == 'phase1_final_leakage_audit_recorded'
assert leakage_summary.get('leakage_audit_ready') is True
assert leakage_validation.get('status') == 'phase1_final_leakage_audit_validation_passed'
assert leakage_audit.get('outer_test_subject_used_in_any_fit') is False
assert leakage_audit.get('test_time_privileged_or_teacher_outputs_allowed') is False
assert leakage_audit.get('runtime_comparator_logs_audited') is False
assert readiness_summary.get('status') == 'phase1_final_comparator_runner_readiness_recorded'
assert readiness_summary.get('upstream_manifests_ready') is True
assert readiness_summary.get('final_comparator_outputs_present') is False
assert readiness_summary.get('runtime_comparator_logs_audited') is False
assert readiness_summary.get('smoke_artifacts_promoted') is False
assert readiness_claim.get('claim_ready') is False


In [ ]:
# Cell 4 - Dataset payload and source hash inventory.

assert DATASET_ROOT.exists(), f'Dataset root missing: {DATASET_ROOT}'
edf_files = sorted(DATASET_ROOT.glob('sub-*/ses-*/eeg/*_eeg.edf'))
events_files = sorted(DATASET_ROOT.glob('sub-*/ses-*/eeg/*_events.tsv'))
print('EDF files discovered:', len(edf_files))
print('Events files discovered:', len(events_files))
assert edf_files, 'No EDF payloads found. Materialize dataset payloads before final feature matrix extraction.'
assert events_files, 'No events.tsv files found. Materialize dataset metadata before extraction.'

HASH_TARGETS = [
    'configs/phase1/final_feature_matrix.json',
    'configs/phase1/final_feature_manifest.json',
    'src/phase1/final_feature_matrix.py',
    'src/phase1/final_feature_manifest.py',
    'src/phase1/final_comparator_runner_readiness.py',
    'src/phase05/estimators.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_feature_matrix.py',
    'notebooks/23_colab_phase1_final_feature_matrix.ipynb',
]
source_hashes = {}
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    assert path.exists(), f'Missing source target: {rel}'
    source_hashes[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(source_hashes, indent=2))


In [ ]:
# Cell 5 - Run the CLI-backed final feature matrix materialization.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_feature_matrix',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--final-split-run', str(FINAL_SPLIT_RUN),
    '--final-feature-run', str(FINAL_FEATURE_RUN),
    '--final-leakage-run', str(FINAL_LEAKAGE_RUN),
    '--runner-readiness-run', str(RUNNER_READINESS_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--matrix-config', 'configs/phase1/final_feature_matrix.json',
]
run(cmd, cwd=REPO_DIR)
print('Final feature matrix materialization command completed. Review artifacts before downstream runner use.')


In [ ]:
# Cell 6 - Inspect latest output and verify required artifacts.

latest = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest.exists())
assert latest.exists(), 'No latest.txt written for final feature matrix output.'
run_dir = Path(latest.read_text(encoding='utf-8').strip())
print('Latest final feature matrix output:', run_dir)
print('Notebook fix marker:', NOTEBOOK_FIX_MARKER)
assert run_dir.exists(), f'Output run directory does not exist: {run_dir}'

required = [
    'phase1_final_feature_matrix_inputs.json',
    'phase1_final_feature_matrix_summary.json',
    'phase1_final_feature_matrix_report.md',
    'phase1_final_feature_matrix_source_links.json',
    'phase1_final_feature_matrix_input_validation.json',
    'phase1_final_feature_matrix_schema.json',
    'final_feature_row_index.json',
    'phase1_final_feature_matrix_validation.json',
    'phase1_final_feature_matrix_claim_state.json',
]
for name in required:
    path = run_dir / name
    print(name, 'exists =', path.exists())
    assert path.exists(), f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_feature_matrix_summary.json')
input_validation = read_json(run_dir / 'phase1_final_feature_matrix_input_validation.json')
validation = read_json(run_dir / 'phase1_final_feature_matrix_validation.json')
schema = read_json(run_dir / 'phase1_final_feature_matrix_schema.json')
row_index = read_json(run_dir / 'final_feature_row_index.json')
claim_state = read_json(run_dir / 'phase1_final_feature_matrix_claim_state.json')
print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'feature_matrix_ready': summary.get('feature_matrix_ready'),
    'n_rows': summary.get('n_rows'),
    'n_expected_rows': summary.get('n_expected_rows'),
    'n_features': summary.get('n_features'),
    'n_expected_features': summary.get('n_expected_features'),
    'skipped_sessions_count': summary.get('skipped_sessions_count'),
    'read_fallbacks_count': summary.get('read_fallbacks_count'),
    'invalid_window_rows_count': summary.get('invalid_window_rows_count'),
    'nonfinite_feature_values': summary.get('nonfinite_feature_values'),
    'missing_source_channels_count': summary.get('missing_source_channels_count'),
    'nonfinite_signal_rows_count': summary.get('nonfinite_signal_rows_count'),
    'bandpower_nonfinite_feature_count': summary.get('bandpower_nonfinite_feature_count'),
    'nonfinite_feature_examples': summary.get('nonfinite_feature_examples'),
    'claim_ready': summary.get('claim_ready'),
    'claim_blockers': summary.get('claim_blockers'),
}, indent=2))

if summary.get('status') != 'phase1_final_feature_matrix_materialized':
    print('Blocked validation details:')
    print(json.dumps({
        'blockers': validation.get('blockers'),
        'nonfinite_feature_values': validation.get('nonfinite_feature_values'),
        'nonfinite_feature_examples': validation.get('nonfinite_feature_examples'),
        'missing_source_channels_count': validation.get('missing_source_channels_count'),
        'missing_source_channels_first10': validation.get('missing_source_channels', [])[:10],
        'missing_feature_counts': validation.get('missing_feature_counts'),
        'nonfinite_signal_rows_count': validation.get('nonfinite_signal_rows_count'),
        'nonfinite_signal_rows_first10': validation.get('nonfinite_signal_rows', [])[:10],
        'bandpower_nonfinite_counts': validation.get('bandpower_nonfinite_counts'),
        'invalid_window_rows': validation.get('invalid_window_rows')[:10],
        'skipped_sessions': validation.get('skipped_sessions')[:10],
    }, indent=2))

# Cell 6 blocked diagnostic preview. This prints before Cell 7 assertions.
if summary.get('status') != 'phase1_final_feature_matrix_materialized':
    print('Cell 6 blocked diagnostic preview:')
    print(json.dumps({
        'feature_matrix_blockers': summary.get('feature_matrix_blockers'),
        'nonfinite_feature_values': summary.get('nonfinite_feature_values'),
        'nonfinite_feature_examples': validation.get('nonfinite_feature_examples'),
        'missing_source_channels_count': validation.get('missing_source_channels_count'),
        'missing_source_channels_first10': validation.get('missing_source_channels', [])[:10],
        'missing_feature_counts': validation.get('missing_feature_counts'),
        'nonfinite_signal_rows_count': validation.get('nonfinite_signal_rows_count'),
        'nonfinite_signal_rows_first10': validation.get('nonfinite_signal_rows', [])[:10],
        'bandpower_nonfinite_counts': validation.get('bandpower_nonfinite_counts'),
        'invalid_window_rows_first10': validation.get('invalid_window_rows', [])[:10],
        'skipped_sessions_first10': validation.get('skipped_sessions', [])[:10],
    }, indent=2))


In [ ]:
# Cell 7 - Feature matrix assertions.
print('Cell 7 diagnostic marker:', NOTEBOOK_FIX_MARKER)

if summary.get('status') != 'phase1_final_feature_matrix_materialized':
    diagnostic_payload = {
        'summary_status': summary.get('status'),
        'feature_matrix_ready': summary.get('feature_matrix_ready'),
        'feature_matrix_blockers': summary.get('feature_matrix_blockers'),
        'n_candidate_rows': summary.get('n_candidate_rows'),
        'n_expected_rows': summary.get('n_expected_rows'),
        'n_features': summary.get('n_features'),
        'n_expected_features': summary.get('n_expected_features'),
        'skipped_sessions_count': summary.get('skipped_sessions_count'),
        'read_fallbacks_count': summary.get('read_fallbacks_count'),
        'invalid_window_rows_count': summary.get('invalid_window_rows_count'),
        'nonfinite_feature_values': summary.get('nonfinite_feature_values'),
        'nonfinite_feature_examples': validation.get('nonfinite_feature_examples'),
        'missing_source_channels_count': validation.get('missing_source_channels_count'),
        'missing_source_channels_first10': validation.get('missing_source_channels', [])[:10],
        'missing_feature_counts': validation.get('missing_feature_counts'),
        'nonfinite_signal_rows_count': validation.get('nonfinite_signal_rows_count'),
        'nonfinite_signal_rows_first10': validation.get('nonfinite_signal_rows', [])[:10],
        'bandpower_nonfinite_counts': validation.get('bandpower_nonfinite_counts'),
        'invalid_window_rows_first10': validation.get('invalid_window_rows', [])[:10],
        'skipped_sessions_first10': validation.get('skipped_sessions', [])[:10],
        'run_dir': str(run_dir),
        'validation_artifact': str(run_dir / 'phase1_final_feature_matrix_validation.json'),
        'blocked_artifact': str(run_dir / 'phase1_final_feature_matrix_blocked.json'),
    }
    print('Final feature matrix materialization is blocked. Diagnostic payload:')
    print(json.dumps(diagnostic_payload, indent=2))
    raise AssertionError(diagnostic_payload)

assert summary.get('status') == 'phase1_final_feature_matrix_materialized', summary.get('feature_matrix_blockers')
assert summary.get('feature_matrix_ready') is True
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert summary.get('contains_model_outputs') is False
assert summary.get('contains_logits') is False
assert summary.get('contains_metrics') is False
assert summary.get('n_rows') == summary.get('n_expected_rows')
assert summary.get('n_features') == summary.get('n_expected_features')
assert summary.get('skipped_sessions_count') == 0
assert summary.get('nonfinite_feature_values') == 0
assert input_validation.get('status') == 'phase1_final_feature_matrix_inputs_ready'
assert validation.get('status') == 'phase1_final_feature_matrix_validation_passed'
assert validation.get('feature_matrix_ready') is True
assert validation.get('feature_names_match_manifest') is True
assert validation.get('contains_model_outputs') is False
assert validation.get('contains_logits') is False
assert validation.get('contains_metrics') is False
assert validation.get('blockers') == []
assert schema.get('feature_names') == feature_manifest.get('feature_names')
assert schema.get('matrix_boundary', {}).get('standalone_claim_ready') is False
assert row_index.get('n_rows') == summary.get('n_rows')
assert 'final_comparator_outputs_missing' in claim_state.get('blockers', [])
assert 'runtime_comparator_leakage_logs_missing_until_final_runners_execute' in claim_state.get('blockers', [])

matrix_path = Path(summary['matrix_path'])
assert matrix_path.exists(), f'Missing final feature matrix CSV: {matrix_path}'
print('Matrix CSV:', matrix_path)
print('Feature matrix materialization assertions passed.')


In [ ]:
# Cell 8 - Lightweight CSV structure check without printing feature values.

import csv

with matrix_path.open(encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    header = reader.fieldnames or []
    first_rows = [next(reader, None) for _ in range(3)]
first_rows = [row for row in first_rows if row is not None]
print('CSV column count:', len(header))
print('Identity columns present:', [name for name in ['row_id', 'participant_id', 'session_id', 'trial_id', 'label', 'set_size'] if name in header])
print('First row identities only:', [{k: row[k] for k in ['row_id', 'participant_id', 'session_id', 'trial_id', 'label', 'set_size']} for row in first_rows])
assert 'logit' not in ''.join(header).lower()
assert 'metric' not in ''.join(header).lower()
assert set(feature_manifest['feature_names']).issubset(set(header))


In [ ]:
# Cell 9 - Write a non-claim review note.

from datetime import datetime, timezone

review_note = {
    'status': 'phase1_final_feature_matrix_review_pass_non_claim',
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'reviewed_run': str(run_dir),
    'scope': 'Phase 1 final scalp feature matrix materialization only',
    'checks_passed': [
        'required_artifacts_present',
        'input_validation_passed',
        'matrix_validation_passed',
        'row_count_matches_final_feature_manifest',
        'feature_names_match_final_feature_manifest',
        'no_skipped_sessions',
        'all_feature_values_finite',
        'contains_model_outputs_false',
        'contains_logits_false',
        'contains_metrics_false',
        'claim_ready_false',
        'headline_phase1_claim_open_false',
    ],
    'n_rows': summary.get('n_rows'),
    'n_features': summary.get('n_features'),
    'subjects': summary.get('subjects'),
    'read_fallbacks_count': summary.get('read_fallbacks_count'),
    'matrix_path': summary.get('matrix_path'),
    'claim_blockers': summary.get('claim_blockers'),
    'allowed_interpretation': claim_state.get('allowed_interpretation'),
    'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    'next_allowed_steps': [
        'Use this final feature matrix as an input to final comparator runner implementation only.',
        'Require final comparator runners to write fold logs, logits, subject-level metrics, runtime leakage logs and output manifests.',
        'Do not treat the feature matrix as decoder efficacy evidence or privileged-transfer evidence.',
        'Do not open headline claims until final comparator outputs, controls, calibration, influence and reporting all pass locked rules.',
    ],
}
review_path = run_dir / 'phase1_final_feature_matrix_review_note.json'
review_path.write_text(json.dumps(review_note, indent=2), encoding='utf-8')
print('Review note written:', review_path)
print(json.dumps(review_note, indent=2))


In [ ]:
# Cell 10 - Closeout.

print('================ Phase 1 Final Feature Matrix Closeout ================')
print('Notebook fix marker: phase1_final_feature_matrix_v2_blocked_diagnostics_20260421')
print('Latest final feature matrix output:', run_dir)
print('Feature matrix ready:', summary.get('feature_matrix_ready'))
print('Rows:', summary.get('n_rows'))
print('Expected rows:', summary.get('n_expected_rows'))
print('Features:', summary.get('n_features'))
print('Skipped sessions:', summary.get('skipped_sessions_count'))
print('Read fallbacks:', summary.get('read_fallbacks_count'))
print('Invalid window rows:', summary.get('invalid_window_rows_count'))
print('Non-finite feature values:', summary.get('nonfinite_feature_values'))
print('Matrix path:', summary.get('matrix_path'))
print('Claim blockers:', summary.get('claim_blockers'))
print('')
print('CHECK REQUIRED: final_feature_matrix.csv contains feature values and labels only. It contains no logits, metrics, model outputs, controls, calibration, influence or runtime leakage logs.')
print('')
print('NOT OK TO CLAIM: This feature matrix notebook does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
